# Environment / Dependencies Setup

The cell below installs all required Python packages:

| Package | Purpose |
|---------|---------|
| `rapidocr`, `onnxruntime` | The OCR engine and the CPU runtime it needs |
| `pymupdf`, `Pillow` | Draw PDF pages as images |
| `sentence-transformers` | The local embedding model |
| `faiss-cpu` | Vector search for Part 2 |
| `langchain-openai` | Connects to the LLM through OpenRouter |
| `requests` | Downloads the sample documents |
| `numpy` | The vector math |
| `python-dotenv` | Loads the API key from `.env` |

In [ ]:
!pip install -q rapidocr onnxruntime pymupdf Pillow sentence-transformers "torch>=2.5" faiss-cpu langchain-openai requests numpy python-dotenv

### Getting an OpenRouter API key

OpenRouter gives you one API key that works across many models, including free ones.

1. Go to [openrouter.ai](https://openrouter.ai) and sign up, or log in.
2. Open the **Keys** section of the dashboard.
3. Click **Create Key**, give it a name, and confirm.
4. **Copy the key immediately**. It is shown in full only once.
5. Put it in a `.env` file next to the notebook:

```
OPENROUTER_API_KEY=sk-or-v1-...
```

The notebook reads this file automatically and asks you to type the key if the file is missing.

---

# Step-wise Instructions — Development

---

### Setup: Import Libraries

In [ ]:
import logging
import os

import faiss
import fitz  # PyMuPDF
import numpy as np
import requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from PIL import Image
from rapidocr import RapidOCR
from sentence_transformers import SentenceTransformer

# Keep the output readable: hide model-download progress bars and noisy library logs
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
for noisy in ("sentence_transformers", "huggingface_hub", "httpx", "transformers"):
    logging.getLogger(noisy).setLevel(logging.ERROR)

### Setup: Load the Key and Create the Three Tools

Three tools are created once here and reused in both parts: the LLM (writes answers), the embedding model (turns text into vectors), and the OCR engine (reads images).

In [ ]:
load_dotenv(".env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key (get one at https://openrouter.ai): ").strip()

# 1. The LLM: a free model on OpenRouter, so running this lab costs nothing
llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    max_tokens=500,
    extra_body={"reasoning": {"enabled": False}},
)

# 2. The embedding model: runs on your CPU, no API key needed
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# 3. The OCR engine: loads its models once; "error" hides the startup chatter
ocr_engine = RapidOCR(params={"Global.log_level": "error"})

print("Key loaded. LLM, embedding model and OCR engine are ready.")

- `temperature=0` asks for the most predictable answer, which is what you want when checking exact figures.
- `base_url` points the OpenAI client at OpenRouter instead of OpenAI. Same client, different server.
- `extra_body={"reasoning": {"enabled": False}}` switches off this model's hidden "thinking" mode. Without it, this reasoning model can spend its whole token budget thinking and return an empty answer.

You should see: `Key loaded. LLM, embedding model and OCR engine are ready.`

---

## Part 1: Short Documents (Whole-Document Retrieval)

**New idea in this part:** OCR that keeps the page layout, one vector per document, and a source tag on every fact in the answer.

### Step 1.1: Download the Sample Documents

A small helper downloads a file and saves it. It is reused in Part 2.

In [ ]:
def download(url, path):
    """Download a file from a URL and save it at the given path."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    response = requests.get(url)
    response.raise_for_status()
    with open(path, "wb") as f:
        f.write(response.content)
    print(f"Downloaded {os.path.basename(path)} ({len(response.content)/1024:.1f} KB)")
    return path

INVOICE_URLS = {
    "simple-invoice.png": "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/simple-invoice.png",
    "contoso-receipt.png": "https://raw.githubusercontent.com/Azure/azure-sdk-for-python/master/sdk/formrecognizer/azure-ai-formrecognizer/tests/sample_forms/receipt/contoso-receipt.png",
    "purchase-order-1.jpg": "https://raw.githubusercontent.com/Azure/azure-sdk-for-python/master/sdk/formrecognizer/azure-ai-formrecognizer/tests/sample_forms/forms/Form_1.jpg",
}

invoice_paths = {}
for filename, url in INVOICE_URLS.items():
    invoice_paths[filename] = download(url, os.path.join("data", "invoices", filename))

You should see three `Downloaded ...` lines.

### Step 1.2: Read Each Document With OCR (Keeping the Layout)

`read_text` runs the OCR engine on one image and returns layout-preserving text. It accepts either a file path or an image array, so Part 2 can reuse it.

In [ ]:
def read_text(image, keep_layout=True):
    """OCR one image (a file path or a numpy array) and return its text."""
    result = ocr_engine(image)
    if keep_layout:
        return result.to_markdown()      # rebuilds rows and columns from the box positions
    return "\n".join(result.txts)        # plain lines, in the order the engine found them

# document_texts maps each filename to its OCR text
document_texts = {}
for filename, path in invoice_paths.items():
    document_texts[filename] = read_text(path)
    print(f"{filename}: {len(document_texts[filename])} characters of text")

print()
print(document_texts["simple-invoice.png"])

The engine is created once and reused. Creating a new `RapidOCR()` for every image would reload its models each time.

You should see the three character counts, then the invoice text with `Invoice Number` and `34278587` in the same column of a small table.

To see what the layout is worth, compare it with the flat version, where the words are simply joined in detection order:

In [ ]:
flat_text = " ".join(ocr_engine(invoice_paths["simple-invoice.png"]).txts)
print(flat_text)

You should see one long line in which every label, date, and amount is mixed together. An LLM reading that line has to guess which number belongs to which header. (Do not test this with "are the label and value on the same line?". That is `True` for the flat text only because flattening puts everything on one line. The property that matters is column alignment, and only the layout version has it.)

### Step 1.3: Embed Each Document (One Vector Each)

Each document is short, so the whole thing becomes one vector. The `embed` helper is reused in Part 2.

In [ ]:
def embed(texts):
    """Turn a list of texts into a 2D array of unit-length vectors, one row per text."""
    return embedder.encode(texts, normalize_embeddings=True)

doc_names = list(document_texts)
doc_vectors = embed([document_texts[name] for name in doc_names])

print(f"Knowledge base built with {len(doc_names)} documents.")
print(f"Vector table shape: {doc_vectors.shape} (documents x dimensions)")

`normalize_embeddings=True` scales every vector to length 1. That makes cosine similarity equal to a plain dot product, which keeps the search code very short.

You should see `Vector table shape: (3, 384)`: three documents, 384 numbers each.

### Step 1.4: Search the Documents

The question is embedded the same way, then compared to every document with a dot product. A higher score means a closer match.

In [ ]:
def search_documents(query, top_k=2):
    """Return the top_k closest documents as {source, text, score}, best first."""
    scores = doc_vectors @ embed([query])[0]          # one cosine score per document
    best = np.argsort(scores)[::-1][:top_k]           # positions of the highest scores
    return [
        {"source": doc_names[i], "text": document_texts[doc_names[i]], "score": float(scores[i])}
        for i in best
    ]

Every search function in this lab returns a list of dictionaries with the same three keys (`source`, `text`, `score`). That shared shape is what lets Part 2 reuse the next two functions unchanged.

### Step 1.5: Ask the LLM and Check the Sources

Two functions that both parts use. `ask_llm` labels each retrieved source and tells the LLM to tag every fact. `show_sources` then looks for those tags in the answer.

In [ ]:
def ask_llm(query, hits):
    """Send the retrieved sources to the LLM and return its tagged answer."""
    context = "\n\n".join(f"[Source: {hit['source']}]\n{hit['text']}" for hit in hits)
    prompt = f"""You answer questions using ONLY the context below.
Every fact you state must be followed by its source tag, copied exactly, like [Source: name].
If the answer is not in the context, say "Not found in the provided documents."

Context:
{context}

Question: {query}
Answer:"""
    return llm.invoke(prompt).content.strip()

def show_sources(answer, hits):
    """Explainability: for every retrieved source, was its tag cited in the answer?"""
    print("--- EXPLAINABILITY ---")
    for hit in hits:
        tag = f"[Source: {hit['source']}]"
        used = tag in answer
        print(f"{hit['score']:.3f}  {hit['source']}  ->  {'USED' if used else 'retrieved but NOT used'}")
        if used:
            for line in answer.splitlines():
                if tag in line:
                    print(f"       cited in: {line.strip()}")

def answer_question(query, hits):
    """Print the retrieved sources, the answer, and the explainability check."""
    print(f"Question: {query}\n")
    print("--- SOURCES RETRIEVED ---")
    for hit in hits:
        print(f"{hit['score']:.3f}  {hit['source']}")
    answer = ask_llm(query, hits)
    print("\n--- ANSWER ---")
    print(answer)
    print()
    show_sources(answer, hits)

- `.invoke()` is LangChain's single call: it sends the prompt and returns a response object. `.content` is the text.
- A retrieved source is not necessarily a used source. The search returns the top 2, so if only one is relevant, the other is retrieved and then ignored. `show_sources` tells the two cases apart by looking for the exact tag.

### Step 1.6: Ask Questions

In [ ]:
query = "What is the invoice number and total charge on the Contoso invoice?"
answer_question(query, search_documents(query))

You should see `simple-invoice.png` ranked first, an answer containing `34278587` and `$56,651.49` with a `[Source: simple-invoice.png]` tag, and `USED` next to that file.

Now a question that should favour a different document:

In [ ]:
follow_up = "Which items were purchased on the Surface Pro receipt?"
answer_question(follow_up, search_documents(follow_up, top_k=3))

`top_k=3` shows every document's score. You should see `contoso-receipt.png` move to the top, and the answer name the Surface Pro and the Surface Pen. If the top document did not change between the two questions, retrieval would not be telling your documents apart.

---

## Part 2: A Long Scanned PDF (Chunks + FAISS)

**New idea in this part:** a long document cannot be one vector, so it is cut into overlapping chunks, indexed in FAISS, and several chunks are retrieved per question. It reuses `download`, `read_text`, `embed`, `ask_llm`, `show_sources`, and `answer_question` from Part 1.

### Step 2.1: Download the PDF and Draw Each Page as an Image

OCR reads images, not PDFs, so every page is first rendered to an image.

In [ ]:
PDF_PATH = download(
    "https://raw.githubusercontent.com/jamalmazrui/pdf2ocr/master/debate.pdf",
    os.path.join("data", "debate.pdf"),
)

def render_pages(pdf_path, dpi=200):
    """Draw every PDF page as an image array. A higher dpi means a sharper image for OCR."""
    zoom = dpi / 72                                   # PDF pages are 72 dpi by default
    matrix = fitz.Matrix(zoom, zoom)
    pages = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            pix = page.get_pixmap(matrix=matrix)
            image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            pages.append(np.array(image))
    return pages

page_images = render_pages(PDF_PATH)
print(f"Rendered {len(page_images)} pages. Page 1 is {page_images[0].shape[1]}x{page_images[0].shape[0]} pixels.")

You should see `Rendered 32 pages`.

### Step 2.2: OCR Every Page

The same `read_text` from Part 1 is used, with `keep_layout=False`. This document is flowing prose, not a table, so plain lines in reading order are what we want (the layout version would add blank lines between every line and waste chunk space).

In [ ]:
page_texts = []
for page_number, image in enumerate(page_images, start=1):
    page_texts.append(read_text(image, keep_layout=False))
    if page_number <= 3:
        print(f"Page {page_number} transcribed: {len(page_texts[-1])} characters")

print(f"... all {len(page_texts)} pages done, {sum(len(t) for t in page_texts)} characters in total.")
print()
print(page_texts[3][:350])

This takes about a minute on a CPU. You should see per-page character counts and the start of page 4, which lists the debate rules. Scanned text is rarely perfect, so expect an occasional garbled word.

### Step 2.3: Cut the Text into Overlapping Chunks

Each chunk is up to 500 characters, and the next chunk starts 450 characters later, so neighbours share 50 characters. Every chunk gets a `source` label with its page and chunk number, so answers can cite it.

In [ ]:
CHUNK_SIZE = 500
OVERLAP = 50
STEP = CHUNK_SIZE - OVERLAP

chunks = []
for page_number, text in enumerate(page_texts, start=1):
    for start in range(0, len(text), STEP):
        piece = text[start : start + CHUNK_SIZE].strip()
        if piece:
            chunks.append({
                "source": f"debate.pdf, page {page_number}, chunk {len(chunks) + 1}",
                "page": page_number,
                "text": piece,
            })

print(f"Total chunks created: {len(chunks)}")
print("End of chunk 1:  ", repr(chunks[0]["text"][-50:]))
print("Start of chunk 2:", repr(chunks[1]["text"][:50]))
print("Chunk 2 starts with text already inside chunk 1:", chunks[1]["text"][:40] in chunks[0]["text"])

You should see the end of chunk 1 and the start of chunk 2 show the same text, and `True` on the last line. That shared text is the overlap. (The two printed strings can differ by a character or two because blank space at the ends of a chunk is trimmed.)

### Step 2.4: Build the FAISS Index

Every chunk is embedded with the same `embed` helper and added to a FAISS index.

In [ ]:
chunk_vectors = embed([chunk["text"] for chunk in chunks]).astype("float32")  # FAISS needs float32

index = faiss.IndexFlatIP(chunk_vectors.shape[1])   # IP = inner product (dot product)
index.add(chunk_vectors)

print(f"Indexed {index.ntotal} chunks, {chunk_vectors.shape[1]} dimensions each.")

Because the vectors have length 1, the inner product FAISS computes is exactly the cosine similarity from Part 1, so scores in both parts mean the same thing. You should see `Indexed ... chunks, 384 dimensions each`.

### Step 2.5: Search the Index (Several Chunks per Question)

In [ ]:
def search_chunks(query, top_k=3):
    """Return the top_k closest chunks as {source, text, score}, best first."""
    query_vector = embed([query]).astype("float32")
    scores, positions = index.search(query_vector, top_k)
    return [
        {**chunks[i], "score": float(score)}
        for score, i in zip(scores[0], positions[0])
    ]

The returned dictionaries have the same `source`, `text`, and `score` keys as Part 1, so the Part 1 functions work on them directly.

### Step 2.6: Ask Questions With an Explainable Answer

In [ ]:
query = "How long does each debate last, and are opening statements allowed?"
answer_question(query, search_chunks(query))

You should see three chunks retrieved, an answer stating 90 minutes and no opening statements (a two-minute closing statement), and the explainability list marking which chunks were cited and which were retrieved but ignored.

A question whose answer is spread over more than one chunk:

In [ ]:
query = "What are the rules about props, notes and charts during a debate?"
answer_question(query, search_chunks(query, top_k=4))

With `top_k=4` the LLM sees four chunks at once and may cite more than one. This is the point of multi-chunk retrieval: the facts needed for an answer are often split across neighbouring chunks, and the explainability check shows exactly which ones the answer drew from.

---

## Part 3: Whole Documents or Chunks?

**New idea in this part:** measure why Part 1 can use whole documents and Part 2 cannot.

An embedding model can only read a limited amount of text per input and silently ignores the rest. This cell compares the retrieval units of both parts to that limit.

In [ ]:
def count_tokens(text):
    """How many word pieces ('tokens') the embedding model needs for this text."""
    return len(embedder.tokenizer.tokenize(text))

print(f"The embedding model reads at most {embedder.max_seq_length} tokens per text.\n")

print("Part 1 units (whole documents):")
for name, text in document_texts.items():
    print(f"  {name}: {len(text)} chars, {count_tokens(text)} tokens")

chunk_tokens = [count_tokens(chunk["text"]) for chunk in chunks]
print(f"\nPart 2 units (chunks): {len(chunks)} chunks, longest = {max(chunk_tokens)} tokens")

whole_pdf = "\n".join(page_texts)
print(f"\nThe whole PDF as ONE unit would be {count_tokens(whole_pdf)} tokens, "
      f"so about {100 - 100 * embedder.max_seq_length // count_tokens(whole_pdf)}% of it would never be seen by the embedding model.")

You should see the invoices fitting comfortably under the limit (the purchase order is the longest and may be close to it), every chunk well under it, and the whole PDF far over it. That is why short documents can be one vector each, while long documents must be chunked.

| | Whole-document retrieval (Part 1) | Chunk retrieval (Part 2) |
|---|---|---|
| **Best for** | A few short documents (invoices, receipts, forms) | Long documents and big collections |
| **Search unit** | One document | A 500-character piece |
| **What the LLM receives** | Complete documents | A few focused passages |
| **Citation points to** | A file | A page and chunk |
| **Search tool** | NumPy dot product | FAISS index |
| **Weak spot** | Anything past the embedding limit is ignored | An answer split across chunks needs several of them retrieved |

---

# What We Learnt

This lab turned scanned images into something you can question, first with short documents and then with a long PDF, and showed which sources ended up in every answer.

- **OCR gives you text from images, and layout matters.** RapidOCR's `to_markdown()` keeps rows and columns together, so `Invoice Number` stays next to `34278587` instead of dissolving into a flat word list.
- **Short documents can be one vector each.** With a handful of invoice-length documents, one embedding per document and a plain dot product retrieve the right one, with no database.
- **Long documents must be chunked.** A scanned PDF is rendered to page images, OCR'd page by page, and cut into 500-character chunks that overlap by 50, so a sentence on a boundary survives whole in at least one chunk.
- **FAISS makes chunk search scale.** The same normalized vectors and the same cosine score are used in both parts; only the search tool changes.
- **Retrieval returns several sources, and the answer is checkable.** The LLM tags every fact with `[Source: ...]`, and looking for that exact tag shows which sources were used and which were retrieved but ignored.
- **The retrieval unit is a design choice.** The embedding model only reads a limited number of tokens, which is why short documents work whole and long ones need chunks.